In [ ]:
import sys
sys.path.append("../")
%load_ext autoreload
%autoreload 2

In [ ]:
from src.toy_problem_to_json import toy_problem_to_json
from src.quadratic_model_lin_approx import quadratic_model_lin_approx
from src.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate
from src.classical_solve_lin_program import solve_lp
import matplotlib.pyplot as plt
import numpy as np

true_false = (True, False)

This notebook is for experimenting on with linear approximations to the toy problem or the GasNet-11 problem.

You can download the relevant GasNet-11 files from Sergey's emails.

Some conclusions:
- The condition number is stable with the bound constraints for the toy problem but goes to infinity without them (is there an eigenvalue of 0?)
- The condition number is a bit higher in general for the real problem but still manageable without the bound constraints.
- Most bound constraint parameters to the linear approximation don't affect the condition number. Some (N, lambda) are hard to check because the toy problem seems to have odd properties compared to the real problem. The condition number is linearly proportional to the number of linear constraints. The effect of if the coefficient is included in the surrogate is large and very different depending on toy vs. real.

In [ ]:
# Number of copies of the network in the toy system
N=20

# Whether the production and flow variables have upper bounds
X_UPPER_BOUNDS=True
F_UPPER_BOUNDS=True

# Capacity / demand parameters
GAMMA=1.0
# GAMMA=3.0
LAMBD=1.0
EPS=0.1
DELTA=0.1

# Lower bounds and upper bound on pressure
LX=LP=LF=0.0
UP=5.0

# The number of linear constraints to use per quadratic constraint
LIN_CONSTRAINTS = 10

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

In [ ]:
errors = []
times = []
n_range = range(1, 21)
# n_range = range(1, 101, 50)
for n in n_range:
    toy_problem_to_json(n, X_UPPER_BOUNDS, F_UPPER_BOUNDS, GAMMA, LAMBD,
                        EPS, DELTA, LX, LP, LF, UP)
    # quadratic_model_lin_approx(LIN_CONSTRAINTS, OUTER_APPROXIMATION, COEFFICIENT_SURROGATE,
    #                             SURROGATE_BOUND_BELOW, SURROGATE_BOUND_ABOVE)
    quadratic_lin_approx_no_surrogate(LIN_CONSTRAINTS, OUTER_APPROXIMATION)
    val, time = solve_lp()
    errors.append(val - n)
    times.append(time)

fig, (ax1, ax2) = plt.subplots(2, 1)
fig.supxlabel("N")
ax1.plot(n_range, errors, label="Error in final objective values")
ax2.plot(n_range, times, label="Solve times")
fig.legend()

In [ ]:
errors = []
times = []
lin_cons_range = range(1, 15)
for lin_cons in lin_cons_range:
    toy_problem_to_json(N, X_UPPER_BOUNDS, F_UPPER_BOUNDS, GAMMA, LAMBD,
                        EPS, DELTA, LX, LP, LF, UP)
    quadratic_model_lin_approx(lin_cons, OUTER_APPROXIMATION, COEFFICIENT_SURROGATE,
                                SURROGATE_BOUND_BELOW, SURROGATE_BOUND_ABOVE)
    # quadratic_lin_approx_no_surrogate(lin_cons, OUTER_APPROXIMATION)
    val, time = solve_lp()
    errors.append(val - N)
    times.append(time)

fig, (ax1, ax2) = plt.subplots(2, 1)
fig.supxlabel("lin constraints")
ax1.plot(lin_cons_range, errors, label="Error in final objective values")
ax2.plot(lin_cons_range, times, label="Solve times")
fig.legend()